# 06 Flask and FastAPI Deployment

## 📚 Learning Objectives

By completing this notebook (~20 min), you will:
- Serve a **simple model** (or dummy predictor) via a **REST API** using **FastAPI** (or Flask)
- Call the API with a **HTTP request** (e.g. from the notebook or curl) and get a **prediction**
- See why we use an API (instead of only running the model in a script) so other services can call it

---

## 🌍 Real life

**Where is this used?** Flask and FastAPI are used to **expose ML models** as **REST APIs** in production so mobile apps, web apps, or other services can send data and get predictions.

**In this notebook we use** **FastAPI** (or a minimal Flask example) to create a **single endpoint** that accepts input and returns a prediction. We use an **API** (instead of only running the model in a notebook) **because** in production we need **HTTP** access so other systems can call the model.

---

**Before starting:** Run the imports cell. Install with `pip install fastapi uvicorn` (or `pip install flask`). For a full server you would run uvicorn in a separate terminal; here we show the **code** that defines the API.

**📌 Covers slide(s):** None — Unit 5 (deployment) has no institution slides; use examples in file order.


## Theory (short)

- **REST API:** Client sends **HTTP request** (e.g. POST with JSON body); server runs the model and returns **JSON response** (e.g. predicted class).
- **Flask:** Lightweight Python web framework; we define routes (e.g. `/predict`) and return JSON.
- **FastAPI:** Modern framework with **automatic docs**, **validation**; good for ML APIs. We use **FastAPI** (instead of only Flask) for faster development and built-in validation (optional).
- **We use an API** so the model can be called over the network by other applications.

## 📥 Inputs & 📤 Outputs

**Inputs:** FastAPI (or Flask), NumPy. We define a **dummy predictor** (or load a small saved model) and one **POST /predict** endpoint.

**Dataset:** Synthetic — dummy predictor (no dataset download; API demo).

**Outputs:** Code that defines the API; instructions to run the server and call the endpoint (e.g. `curl` or `requests`). In a notebook we can't run the server in the background easily, so we show the **app code** and a **test request** using a test client.

**Expected:** Running the test cell should return a JSON response (e.g. {"prediction": ...}). When you run the server locally, a POST to /predict should return the same structure.

## Step 1: Imports and define a dummy model (we use a simple function so the API code runs without loading a large model)

In [1]:
# Step 1: Dummy predictor (replace with model.load + model.predict in production)
import numpy as np

def predict_dummy(features: list) -> dict:
    """Dummy predictor: in production replace with model.predict()."""
    arr = np.array(features, dtype=np.float32)
    pred = int(np.clip(np.sum(arr) > 0, 0, 1))
    return {"prediction": pred, "input_sum": float(np.sum(arr))}

print("Dummy predictor ready. In production we would load model and call model.predict(x).")

Dummy predictor ready. In production we would load model and call model.predict(x).


## Step 2: Define FastAPI app (we use FastAPI instead of only Flask for automatic docs and validation)

In [2]:
# Step 2: Define FastAPI app and /predict endpoint
try:
    from fastapi import FastAPI
    from pydantic import BaseModel
    HAS_FASTAPI = True
except ImportError:
    HAS_FASTAPI = False

if HAS_FASTAPI:
    app = FastAPI(title="ML API", description="Simple prediction endpoint")

    class RequestBody(BaseModel):
        features: list[float]

    @app.post("/predict")
    def predict(body: RequestBody):
        out = predict_dummy(body.features)
        return out

    print("FastAPI app defined. Run in terminal: uvicorn <module>:app --reload")
    print("Then POST to http://127.0.0.1:8000/predict with JSON: {\"features\": [0.1, -0.2, 0.3]}")
else:
    print("Install: pip install fastapi uvicorn")

Install: pip install fastapi uvicorn


## Step 3: Test the predictor directly (same logic the API would run)

In [3]:
# Step 3: Test predictor (same logic the API would run)
out = predict_dummy([0.1, -0.2, 0.3])
print("Test predict_dummy([0.1, -0.2, 0.3]):", out)
print("\nWhen the server is running, a client would send: POST /predict with body {\"features\": [0.1, -0.2, 0.3]} and get this JSON back.")

Test predict_dummy([0.1, -0.2, 0.3]): {'prediction': 1, 'input_sum': 0.20000001788139343}

When the server is running, a client would send: POST /predict with body {"features": [0.1, -0.2, 0.3]} and get this JSON back.


## 🌍 Real-World Worked Example — Deploy a Trained Model as a REST API

**Industry context:**
- Spotify's recommendation model is served via a FastAPI microservice handling 400M users
- Instagram's image moderation runs as a containerised PyTorch model behind a REST endpoint
- Every ML feature in a modern app goes through a model serving layer like this

We train a small classifier, export it, and build a **FastAPI endpoint** you can call with curl.

In [4]:
# ── Part 1: Train and save a model ────────────────────────────────────────
import torch, torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

iris = load_iris()
X = StandardScaler().fit_transform(iris.data.astype(np.float32))
y = iris.target
X_tr,X_te,y_tr,y_te = train_test_split(X, y, test_size=0.2, random_state=42)

model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
opt   = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
Xt = torch.tensor(X_tr); Yt = torch.tensor(y_tr, dtype=torch.long)

for _ in range(200):
    loss = loss_fn(model(Xt), Yt)
    opt.zero_grad(); loss.backward(); opt.step()

torch.save(model.state_dict(), '/tmp/iris_model.pt')
print("Model saved to /tmp/iris_model.pt")

# Verify
model.eval()
with torch.no_grad():
    acc = (model(torch.tensor(X_te)).argmax(1)==torch.tensor(y_te)).float().mean()
print(f"Test accuracy: {acc:.2%}")

# ── Part 2: Simulate the FastAPI serving code ─────────────────────────────
# (In production, save this as main.py and run: uvicorn main:app --reload)
fastapi_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import torch, torch.nn as nn
import numpy as np

app = FastAPI(title="Iris Classifier API")

# Load model at startup
model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
model.load_state_dict(torch.load("/tmp/iris_model.pt"))
model.eval()
CLASSES = ["setosa", "versicolor", "virginica"]

class IrisRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.post("/predict")
def predict(req: IrisRequest):
    features = torch.tensor([[req.sepal_length, req.sepal_width,
                               req.petal_length, req.petal_width]])
    with torch.no_grad():
        logits = model(features)
        probs  = torch.softmax(logits, dim=1)[0]
        label  = CLASSES[probs.argmax().item()]
    return {"prediction": label, "confidence": round(probs.max().item(), 3)}

@app.get("/health")
def health(): return {"status": "ok"}

# Run with: uvicorn main:app --host 0.0.0.0 --port 8000
# Test with: curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" \
#            -d '{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'
'''
print("\n── FastAPI serving code (save as main.py) ────────────────────────────────")
print(fastapi_code)
print("\nThis is exactly how Spotify and Uber serve their ML models in production.")

Model saved to /tmp/iris_model.pt
Test accuracy: 96.67%

── FastAPI serving code (save as main.py) ────────────────────────────────

from fastapi import FastAPI
from pydantic import BaseModel
import torch, torch.nn as nn
import numpy as np

app = FastAPI(title="Iris Classifier API")

# Load model at startup
model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
model.load_state_dict(torch.load("/tmp/iris_model.pt"))
model.eval()
CLASSES = ["setosa", "versicolor", "virginica"]

class IrisRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.post("/predict")
def predict(req: IrisRequest):
    features = torch.tensor([[req.sepal_length, req.sepal_width,
                               req.petal_length, req.petal_width]])
    with torch.no_grad():
        logits = model(features)
        probs  = torch.softmax(logits, dim=1)[0]
        label  = CLASSES[probs.argmax().item()]
    return {"prediction": label, "confi

## 🧩 Mini-exercise

**Try it:** Add a second endpoint (e.g. `GET /health`) that returns `{"status": "ok"}`. Or change the predictor to return both the predicted class and a confidence score.

---

## ✅ Summary

**What you did:** Defined a **dummy predictor** and a **FastAPI app** with a **POST /predict** endpoint that accepts features and returns a prediction. Tested the predictor logic directly.

**In real life you'd also:** Load a real saved model, run the server with `uvicorn`, add authentication, and deploy to a cloud (e.g. GCP, AWS).

**The main idea:** We expose the model as a **REST API** (e.g. FastAPI) so other services can send data and get predictions over HTTP.

**Next (in sequence):** `07_model_optimization_quantization.ipynb` (quantization). **See also:** `01_model_optimization` (save/load), `02_tensorflow_serving` (TensorFlow Serving).

## 📚 References & Further Reading

**Frameworks:**
- [FastAPI Documentation](https://fastapi.tiangolo.com/) — Modern Python API framework
- [ONNX Runtime](https://onnxruntime.ai/) — Cross-platform inference
- [BentoML](https://github.com/bentoml/BentoML) — ML model serving framework

**Cloud Services:**
- [AWS SageMaker Inference](https://docs.aws.amazon.com/sagemaker/latest/dg/deploy-model.html)
- [Google Cloud Vertex AI](https://cloud.google.com/vertex-ai/docs/predictions/overview)

**State-of-the-Art:** Uber, Airbnb, and Spotify deploy hundreds of ML models using microservices with FastAPI/gRPC.